# 02 - Optimización e hiperparámetros

Se compara SGD con Momentum, RMSprop y Adam con el mismo modelo y los mismos datos. Después se evalúan tres learning rates de Adam. Las decisiones se basan en Dev; Test queda reservado.

## 1. Preparación de datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, y_train.shape)
print("Dev:", x_dev.shape, y_dev.shape)
print("Test:", x_test.shape, y_test.shape)
print("Rango de píxeles:", x_train.min(), x_train.max())

## 2. Crear una arquitectura idéntica por experimento

In [ ]:
def crear_modelo(optimizer, units=128):
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)), layers.Flatten(),
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    modelo.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

## 3. Comparar optimizadores

In [ ]:
fabricas = {
    "SGD + Momentum": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": lambda: keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam": lambda: keras.optimizers.Adam(learning_rate=0.001)
}
resultados, historias = [], {}
for nombre, fabrica in fabricas.items():
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    modelo = crear_modelo(fabrica())
    hist = modelo.fit(x_train, y_train, validation_data=(x_dev, y_dev),
                      epochs=5, batch_size=64, verbose=0)
    historias[nombre] = hist
    resultados.append({"optimizador": nombre,
                       "train_accuracy": hist.history["accuracy"][-1],
                       "dev_accuracy": hist.history["val_accuracy"][-1],
                       "dev_loss": hist.history["val_loss"][-1]})
tabla_opt = pd.DataFrame(resultados).sort_values("dev_loss")
display(tabla_opt)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for nombre, hist in historias.items():
    ax.plot(hist.history["val_loss"], marker="o", label=nombre)
ax.set(xlabel="Época", ylabel="Dev loss", title="Comparación de optimizadores")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

## 4. Comparar learning rate de Adam

In [ ]:
resultados_lr, historias_lr = [], {}
for lr in [0.0001, 0.001, 0.01]:
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    modelo = crear_modelo(keras.optimizers.Adam(learning_rate=lr))
    hist = modelo.fit(x_train, y_train, validation_data=(x_dev, y_dev),
                      epochs=3, batch_size=64, verbose=0)
    historias_lr[lr] = hist
    resultados_lr.append({"learning_rate": lr,
                          "train_accuracy": hist.history["accuracy"][-1],
                          "dev_accuracy": hist.history["val_accuracy"][-1],
                          "dev_loss": hist.history["val_loss"][-1]})
tabla_lr = pd.DataFrame(resultados_lr).sort_values("dev_loss")
display(tabla_lr)

fig, ax = plt.subplots(figsize=(8, 5))
for lr, hist in historias_lr.items():
    ax.plot(hist.history["val_loss"], marker="o", label=f"Adam lr={lr}")
ax.set(xlabel="Época", ylabel="Dev loss", title="Efecto del learning rate")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

**Análisis para el PDF:** indica cuál configuración obtuvo menor Dev loss y/o mayor Dev accuracy; compara estabilidad. Explica si el learning rate pequeño aprendió más lento o el mayor fue inestable. La conclusión solo corresponde a las condiciones probadas; Adam no es universalmente superior.